
# 05 — ViT Patch Embedding + VLM Projector, from Scratch

**Goal:** implement the ViT front end (patchify → linear projection → position embedding),
confirm it's mathematically identical to the "conv with kernel=stride=patch_size" formulation
you'll see in real code, then implement and contrast the two dominant families of VLM projector
— MLP-style (LLaVA) and resampler/Q-Former-style — and see directly, via the token-count
behavior, why they trade off differently against image resolution. This directly extends
notebook 1's cross-attention material into the "how does the vision side actually connect to
the LLM" question that's a near-universal VLM interview topic.

Structure: **Lesson → Implementation → Quiz → Final Answers & Explanations.**



## 1. Lesson

### 1.1 From image to patch tokens

A Vision Transformer treats an image the way a language transformer treats a sentence: as a
sequence of tokens. To get there, the image is cut into a grid of non-overlapping
$P \times P$ patches, each patch is flattened into a vector of length $C \cdot P \cdot P$ ($C$ =
channels), and a single shared linear layer projects every flattened patch into the model's
embedding dimension:
$$
\text{patch\_embed}_i = W_{proj} \cdot \text{flatten}(\text{patch}_i) + b, \qquad i = 1, \dots, \frac{H}{P}\cdot\frac{W}{P}
$$
Because patches have no inherent order once flattened into a sequence (unlike text, where token
order is obviously meaningful, a naive transformer has no idea patch 5 is spatially adjacent to
patch 6), a **position embedding** is added per patch — otherwise the model would be permutation-
invariant to how you scan the image into a sequence, throwing away all spatial structure.

### 1.2 The conv equivalence

You'll almost never see "flatten each patch, then apply one shared linear layer" spelled out
literally in real code — instead you'll see a single `Conv2d` with `kernel_size = stride =
patch_size`. These are exactly the same operation: a convolution with a kernel exactly as large
as the stride, applied with no overlap, is mathematically identical to slicing the image into
non-overlapping patches and applying one shared linear layer to each flattened patch — the conv
kernel's weights, reshaped, *are* the linear projection's weight matrix. This is a common
interview "gotcha" — being able to say *why* these are the same operation (not just that they
happen to give the same output) signals real understanding of what a strided conv is actually
doing.

### 1.3 The projector: connecting vision to the LLM

Once you have a sequence of patch embeddings (in the vision encoder's own embedding space), you
need to project them into the **LLM's** embedding space so they can be consumed as (pseudo-)
tokens. Two dominant families, both of which you should be able to implement and contrast:

- **MLP projector (LLaVA-style)**: a small (1-2 layer) MLP applied independently to every patch
  embedding, mapping vision-dim → LLM-dim. Simple, and — crucially — **produces one output token
  per input patch**, so the number of vision tokens fed into the LLM scales directly with image
  resolution (more patches → more tokens → more context length consumed, and $O(L^2)$ attention
  cost inside the LLM grows accordingly).
- **Resampler / Q-Former-style projector**: a small, **fixed** number of learned query tokens
  cross-attend into the (arbitrarily large) set of patch embeddings — exactly the cross-attention
  mechanism from notebook 1, with the patch embeddings as the K/V side and the learned queries as
  the Q side. This produces a **fixed number of output tokens regardless of image resolution**
  (more patches just means the *context* the resampler queries are attending into is larger, not
  that more tokens come out) — but that fixed budget of output tokens is a real information
  bottleneck: a very information-rich, very high-resolution image still has to squeeze through the
  same small number of query slots as a simple one.

This is a direct, concrete trade-off: MLP-style trades context-length cost for simplicity and no
information bottleneck; resampler-style trades a fixed information bottleneck for keeping the
LLM's context length independent of image resolution/count — the same complexity argument
introduced in notebook 1 (section 1.5), now made concrete with actual token counts.

### 1.4 Patch size as its own trade-off

Smaller patches (e.g. 14×14 vs. 16×16 vs. 32×32) preserve more fine-grained spatial detail but
produce more patches (longer sequence, more compute in a self-attention vision encoder, and — for
an MLP-style projector — more LLM context consumed); larger patches are cheaper and shorter but
blur together fine detail within each patch. This is the same "resolution vs. sequence length"
trade familiar from tokenization choices in text, just in the spatial domain.


In [ ]:

import torch
import torch.nn as nn

torch.manual_seed(0)



## 2. Implementation

Four pieces to fill in (`# TODO`):

1. `patchify(images, patch_size)` — slice `(B, C, H, W)` images into `(B, num_patches, C*P*P)`
   flattened patches.
2. `unpatchify(patches, patch_size, C, H, W)` — the exact inverse, used only to sanity-check
   `patchify` (real pipelines don't need this, but round-tripping is the cleanest correctness
   check).
3. `PatchEmbedding` — `patchify` + linear projection + learned position embedding. Also implement
   `forward_via_conv`, an alternate formulation using `nn.Conv2d(kernel_size=stride=patch_size)`
   with the *same* weights, to verify the conv equivalence from section 1.2 directly.
4. `MLPProjector` and `ResamplerProjector` — the two projector families from section 1.3.
   `ResamplerProjector` reuses `nn.MultiheadAttention` (as in notebook 1) with a fixed learned
   query parameter.


In [ ]:

def patchify(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    # images: (B, C, H, W) -> (B, num_patches, C*patch_size*patch_size)
    B, C, H, W = images.shape
    P = patch_size
    assert H % P == 0 and W % P == 0, "H, W must be divisible by patch_size"
    # TODO: reshape to (B, C, H//P, P, W//P, P), then permute to group (H//P, W//P) together and
    # (C, P, P) together, then reshape/flatten to (B, num_patches, C*P*P).
    # Hint: x.reshape(B, C, H//P, P, W//P, P).permute(0, 2, 4, 1, 3, 5).reshape(B, -1, C*P*P)
    raise NotImplementedError


def unpatchify(patches: torch.Tensor, patch_size: int, C: int, H: int, W: int) -> torch.Tensor:
    # exact inverse of patchify -- used only for the round-trip sanity test below
    P = patch_size
    B, num_patches, _ = patches.shape
    nh, nw = H // P, W // P
    # TODO: reshape/permute back to (B, C, H, W). This should exactly mirror patchify in reverse.
    raise NotImplementedError


class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, in_chans: int, embed_dim: int, num_patches: int):
        super().__init__()
        self.patch_size = patch_size
        # TODO: self.proj = nn.Linear(in_chans * patch_size * patch_size, embed_dim)
        # TODO: self.pos_embed = nn.Parameter of shape (1, num_patches, embed_dim), small random init
        self.proj = None
        self.pos_embed = None

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        # TODO: patchify -> linear proj -> add position embedding
        raise NotImplementedError

    def forward_via_conv(self, images: torch.Tensor) -> torch.Tensor:
        \"\"\"
        Alternate formulation: a Conv2d with kernel_size=stride=patch_size, using the SAME
        weights as self.proj (reshaped), to verify it produces identical output to forward().
        \"\"\"
        B, C, H, W = images.shape
        conv = nn.Conv2d(C, self.proj.out_features, kernel_size=self.patch_size, stride=self.patch_size)
        with torch.no_grad():
            # TODO: copy self.proj's weight (reshaped to the conv's (out_ch, in_ch, P, P) shape)
            # and bias into `conv`.
            pass
        # TODO: run images through `conv` -> (B, embed_dim, H/P, W/P), then flatten spatial dims
        # and transpose to (B, num_patches, embed_dim), then add self.pos_embed, return.
        raise NotImplementedError


In [ ]:

class MLPProjector(nn.Module):
    \"\"\"LLaVA-style: one output token per input patch. Token count SCALES with input length.\"\"\"
    def __init__(self, vision_dim: int, llm_dim: int, hidden: int = None):
        super().__init__()
        hidden = hidden or llm_dim
        # TODO: self.net = a 2-layer MLP: Linear(vision_dim, hidden) -> GELU -> Linear(hidden, llm_dim)
        self.net = None

    def forward(self, patch_feats: torch.Tensor) -> torch.Tensor:
        # TODO: apply self.net to every patch token independently -> (B, num_patches, llm_dim)
        raise NotImplementedError


class ResamplerProjector(nn.Module):
    \"\"\"
    Q-Former/Perceiver-Resampler-style: a FIXED number of learned query tokens cross-attend into
    the patch features. Token count is FIXED regardless of input length -- this is the same
    cross-attention mechanism as notebook 1, with patch_feats as the K/V side.
    \"\"\"
    def __init__(self, vision_dim: int, llm_dim: int, num_queries: int, num_heads: int = 4):
        super().__init__()
        self.num_queries = num_queries
        # TODO: self.query = nn.Parameter of shape (1, num_queries, llm_dim), small random init
        # TODO: self.kv_proj = nn.Linear(vision_dim, llm_dim)  (project patch feats into llm_dim
        #       so Q and K/V live in the same space for cross-attention)
        # TODO: self.cross_attn = nn.MultiheadAttention(llm_dim, num_heads, batch_first=True)
        self.query = None
        self.kv_proj = None
        self.cross_attn = None

    def forward(self, patch_feats: torch.Tensor) -> torch.Tensor:
        # TODO: expand self.query to the batch size, project patch_feats via self.kv_proj,
        # run cross-attention (query attends into the projected patch feats as K and V),
        # return the attention output -> (B, num_queries, llm_dim)
        raise NotImplementedError



### Sanity tests

1. **Patchify/unpatchify round trip** — the cleanest possible correctness check for `patchify`.
2. **Conv equivalence** — `PatchEmbedding.forward` and `.forward_via_conv` (same weights) must
   produce identical output.
3. **Token-count behavior** — feed both projectors patch sequences of two different lengths
   (simulating two different image resolutions) and confirm `MLPProjector`'s output length
   scales with input length while `ResamplerProjector`'s stays fixed at `num_queries` either way.
   This is the test that makes section 1.3's trade-off concrete rather than just asserted.


In [ ]:

B, C, H, W, P = 2, 3, 8, 8, 4
imgs = torch.randn(B, C, H, W)

patches = patchify(imgs, P)
assert patches.shape == (B, (H // P) * (W // P), C * P * P)
recon = unpatchify(patches, P, C, H, W)
assert torch.allclose(imgs, recon, atol=1e-6), "unpatchify should exactly invert patchify"
print("Test 1 passed: patchify -> unpatchify round trip is exact")


In [ ]:

embed_dim = 16
num_patches = (H // P) * (W // P)
pe = PatchEmbedding(P, C, embed_dim, num_patches)

out_linear = pe(imgs)
out_conv = pe.forward_via_conv(imgs)
max_diff = (out_linear - out_conv).abs().max().item()
assert torch.allclose(out_linear, out_conv, atol=1e-4), f"conv/linear mismatch, max diff={max_diff}"
print(f"Test 2 passed: conv formulation matches reshape+linear formulation (max diff {max_diff:.2e})")


In [ ]:

vision_dim, llm_dim, num_queries = embed_dim, 24, 8
mlp_proj = MLPProjector(vision_dim, llm_dim)
resampler_proj = ResamplerProjector(vision_dim, llm_dim, num_queries)

patch_feats_small = torch.randn(2, 4, vision_dim)   # e.g. a low-res image: 4 patches
patch_feats_large = torch.randn(2, 64, vision_dim)  # e.g. a high-res image: 64 patches

mlp_small = mlp_proj(patch_feats_small)
mlp_large = mlp_proj(patch_feats_large)
assert mlp_small.shape == (2, 4, llm_dim)
assert mlp_large.shape == (2, 64, llm_dim)
print(f"MLP projector token counts: {mlp_small.shape[1]} (small) vs {mlp_large.shape[1]} (large) -- scales with input")

resampler_small = resampler_proj(patch_feats_small)
resampler_large = resampler_proj(patch_feats_large)
assert resampler_small.shape == (2, num_queries, llm_dim)
assert resampler_large.shape == (2, num_queries, llm_dim)
print(f"Resampler projector token counts: {resampler_small.shape[1]} (small) vs {resampler_large.shape[1]} (large) -- FIXED")

print("\nTest 3 passed: MLP projector's token count scales with input length; "
      "Resampler projector's stays fixed regardless of input length.")



## 3. Quiz

1. Why is a strided convolution with `kernel_size == stride == patch_size` mathematically
   identical to patchify-then-linear-project, not just coincidentally similar in output shape?
2. Why do patch embeddings need an added position embedding at all — what would go wrong (or
   what capability would be lost) without one?
3. What's the concrete downside of the MLP-projector approach as image resolution or the number
   of images in a single prompt grows? What's the concrete downside of the resampler approach,
   even though its token count doesn't grow?
4. In `ResamplerProjector`, which tensor plays the role of Q, and which plays K/V, in the
   cross-attention call? What would happen (conceptually) if you swapped them?
5. How does patch size ($P$) trade off sequence length against spatial detail, and how does that
   interact with the choice of projector (MLP vs. resampler)?
6. Suppose you need a VLM that can ingest many high-resolution images in a single conversation
   without blowing the LLM's context budget. Which projector family would you lean toward, and
   what would you tell a teammate is the cost of that choice?
7. The resampler's learned query tokens are parameters of the *projector*, not the vision encoder
   or the LLM. Why does it make sense for them to live there, and what do they end up learning to
   represent?

*(Your answers here)*



## 4. Final Answers & Explanations

### Q1 — Why the conv equivalence is exact, not coincidental
A convolution slides a kernel over the input and, at each position, computes a dot product
between the kernel's weights and the input region it currently covers. When `stride ==
kernel_size == patch_size`, the kernel never overlaps itself between positions — each "slide"
lands on a completely disjoint, non-overlapping $P\times P$ region, which is *exactly* the same
partition of the image that patchify produces. And the dot product the conv computes at each
position — kernel weights against the flattened patch of pixels under it — is *literally* the
same arithmetic as `flatten(patch) @ W_proj^T` for a linear layer, just with the kernel's weight
tensor reshaped from `(out_ch, in_ch, P, P)` into a `(out_ch, in_ch*P*P)` matrix. So it isn't
that the two approaches happen to agree — reshaping one operation's weights into the other's
shape makes them the identical computation, which is exactly what the notebook's Test 2 confirms
numerically.

### Q2 — Why position embeddings are necessary
Once an image is patchified and flattened into a sequence, a plain self-attention block (or the
resampler's cross-attention) has no built-in notion of which patch was spatially where —
attention over a *set* of tokens is permutation-invariant unless something in the token
representations breaks that symmetry. Without an added position embedding, shuffling the patch
order before feeding them into the model wouldn't change the model's output at all — which would
throw away essentially all spatial structure (a face and a shuffled jumble of the same patches
would look identical to the model). The position embedding is what lets the model learn that
"patch at grid position (2,3) is adjacent to patch at (2,4)," recovering the 2D spatial layout
that flattening into a 1D sequence otherwise destroys.

### Q3 — The concrete trade-off
MLP-style: because it emits one token per patch, the number of vision tokens fed to the LLM
scales directly with resolution/patch count — a handful of high-resolution images (or many
images in one prompt) can consume a large fraction of the LLM's context window before any actual
text appears, and the LLM's own $O(L^2)$ self-attention cost grows correspondingly. Resampler-
style: the token count is fixed regardless of resolution, so context cost stays flat — but that
fixed number of query slots is a genuine information bottleneck; a very detailed, high-resolution
image still has to be compressed into the same small number of output tokens as a simple one,
and information that doesn't fit through that bottleneck is lost, no matter how much the encoder
itself captured.

### Q4 — Q vs. K/V in the resampler
The **learned query tokens** (`self.query`, a fixed small set of parameters) play the role of
$Q$; the **patch embeddings** (projected via `kv_proj`) play the role of $K$ and $V$. This
mirrors exactly the general cross-attention framing from notebook 1: the query side determines
the *output* sequence length (here: `num_queries`, fixed), while the key/value side can be
whatever length the input context happens to be (here: however many patches the image produced).
If you swapped them — patch embeddings as Q, learned tokens as K/V — the output sequence length
would become the number of patches (no longer fixed, defeating the entire purpose of using a
resampler in the first place), and the small set of "query" parameters would play a role more
like a small fixed dictionary being looked up by every patch, which is a fundamentally different
(and not what Q-Former/Resampler architectures are designed to do) mechanism.

### Q5 — Patch size trade-off
Smaller $P$ means each patch covers less spatial area, so fine-grained detail within any single
patch is better preserved — but it also means more patches are needed to cover the same image,
lengthening the token sequence (more compute in the vision encoder's own self-attention, and,
for an MLP-style projector, directly more tokens consumed in the LLM's context). Larger $P$
shortens the sequence and cost but blurs together detail that falls within a single patch's
boundaries (fine text or small objects can become hard to resolve). This interacts with projector
choice: an MLP-style projector inherits patch-size's sequence-length cost directly (smaller
patches → proportionally more LLM tokens), whereas a resampler-style projector's *output* length
is decoupled from patch size (still fixed at `num_queries`) — so a resampler gives you more
freedom to use smaller patches (better detail) without paying additional LLM context cost, at
the price of still funneling that detail through the same fixed bottleneck either way.

### Q6 — Choosing a projector for many high-res images
Given the goal — many high-resolution images without blowing the context budget — a
resampler/Q-Former-style projector is the natural choice, since its output token count per image
stays fixed regardless of resolution, keeping total context cost predictable and bounded even as
image count or resolution grows. The cost to flag to a teammate: every image now gets compressed
through the same small, fixed number of query slots, which is a real information bottleneck —
fine details that would have survived in an MLP-style (one-token-per-patch) approach may be lost,
so this trade only makes sense if the downstream task doesn't need every fine-grained visual
detail preserved token-for-token, or if the resampler is given enough query slots / trained well
enough that the bottleneck isn't the limiting factor in practice.

### Q7 — Why the query tokens live in the projector
The learned query tokens aren't image-specific (they're the same fixed parameters for every
image the model ever processes) and they aren't part of the LLM's own vocabulary/embedding space
either — they exist purely to define *how many* output slots the projector produces and to serve
as the "asking" side of the cross-attention that pulls information out of whatever patch
embeddings this particular image produced. That role is specific to the projector's job (bridging
an arbitrary-length vision representation into a fixed-length one for the LLM to consume), so
it's the natural place for them to be trained parameters. Through training, these query tokens
end up specializing — different queries tend to learn to attend to different kinds of visual
content or spatial regions (e.g., one query slot consistently pulling in "what object is this,"
another leaning toward "what's the overall scene/background," etc.) — effectively learning a
fixed, reusable "basis" of questions to ask any incoming image.
